In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import numpy as np
import pandas as pd
import re
from bs4 import BeautifulSoup

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
df = pd.read_excel('리스크이진분류.xlsx')
df.head(2)

,Unnamed: 0,naver_article_id,writer_nickname,released_at,view_count,title,like_count,comment_count,comments,content,...,log_like_count,log_comment_count,text,is_cartier,is_product,deleted_comment_count,deleted_comment_ratio,label,pred_label,pred_prob
0,0,5536787,2024Y,2025-08-19 23:38:00,345,불가리 뱀이냐 까르띠에 팬더냐 🤣 뭐가 더 이쁜가요?,0,4,앗 전 압도적으로 2번이요|불가리 착용해보심 실물 넘예뻐요!!\n링부분도 투보가스라...,"\n <div class=""se-component se-...",...,0.0,1.609438,불가리 뱀이냐 까르띠에 팬더냐 🤣 뭐가 더 이쁜가요?\n무기명 투표 중 반지 투보가...,1,1,0,0.0,0,0,0.127695
1,1,5536774,가오리맘,2025-08-19 23:22:00,877,2천만원 예산 골라주세요~,0,10,11111111|1번이 뭔가 잘하고 다닐 것 같긴해요ㅎㅎ|1번요|1번이군요 ㅎㅎ|화...,"\n <div class=""se-component se-...",...,0.0,2.397895,2천만원 예산 골라주세요~\n​ 현재 가지고 있는 제품은 모두 로즈골드구요 -반클리...,1,1,0,0.0,0,0,0.002468


In [4]:
df.isnull().sum()

Unnamed: 0                 0
naver_article_id           0
writer_nickname            0
released_at                0
view_count                 0
title                      0
like_count                 0
comment_count              0
comments                 241
content                    0
clean_content              5
log_view_count             0
log_like_count             0
log_comment_count          0
text                       0
is_cartier                 0
is_product                 0
deleted_comment_count      0
deleted_comment_ratio      0
label                      0
pred_label                 0
pred_prob                  0
dtype: int64

In [5]:
def extract_se_text(html):
    soup = BeautifulSoup(html, 'html.parser')
    se_text = soup.find(class_='se-text')
    if se_text:
        text = se_text.get_text(separator=' ').strip()
        text = text.replace('\u200b', '').replace('\xa0', '')
        return text
    else:
        return ''

df["se_text"] = df["content"].apply(extract_se_text)
df["text"] = df["title"].fillna('') + ' ' + df["se_text"].fillna('')
df["text"] = df["text"].str.strip()

In [6]:
df.isnull().sum()

Unnamed: 0                 0
naver_article_id           0
writer_nickname            0
released_at                0
view_count                 0
title                      0
like_count                 0
comment_count              0
comments                 241
content                    0
clean_content              5
log_view_count             0
log_like_count             0
log_comment_count          0
text                       0
is_cartier                 0
is_product                 0
deleted_comment_count      0
deleted_comment_ratio      0
label                      0
pred_label                 0
pred_prob                  0
se_text                    0
dtype: int64

In [7]:
df_risk = df[df['pred_label'] == 1]
print(f"리스크 게시글 개수 : {len(df_risk)}개")

리스크 게시글 개수 : 1860개


In [8]:
# 1. 키워드 사전 정의
risk_keywords = {
    "품질리스크": [
        # 외관 손상
        "스크래치", "기스", "흠집", "파손", "깨짐", "휘어짐",

        # 변색/광택
        "변색", "옐골화", "옐로우골드", "광", "광택",
        "기스광", "쇳덩이", "쇳덩어리", "묻어나", "이염",

        # 도금
        "도금", "벗겨",

        # 시계 기능
        "멈", "느려진", "오작동", "작동 안",

        # 부품
        "체인", "고장",

        # 가품
        "가품", "짝퉁", "짭", "정품",

        # 수리/보수
        "폴리싱", "오버홀",

        # 기타
        "불량", "하자", "이물질", "냄새", "품질",
        "노다이아", "민자",
    ],

    "운영리스크": [
        # 웨이팅/오픈런
        "웨이팅", "대기", "오픈런", "까픈런",

        # 은어
        "까르띠에 대전", "존버",

        # 공홈/온라인
        "공홈", "공식홈페이지", "온라인", "앱", "사이트",
        "새로고침", "새로 고침", "품절",

        # 배송
        "배송", "지연", "오배송", "늦",

        # 입고/재고
        "입고", "재고",

        # AS/수선
        "AS", "수선", "폴리싱", "오버홀", "주문",
    ],

    "평판리스크": [
        # 감정/반응
        "실망", "최악", "별로", "화남", "역대급", "황당",
        "기분나쁨", "기분 상하", "감정 상하",

        # 브랜드 이슈
        "논란", "불매", "블랙리스트",

        # 직원 태도
        "불친절", "직원", "셀러", "매장 직원", "응대", "태도", "갑질", "무시",
        "진상", "싸가지", "경우가 없", "기본이 없", "기본도 없", "기본도 안",
        "일하기 싫", "성의없", "성의 없", "건성", "쌩까",
        "쳐다도 안", "눈도 안", "대꾸도 안", "무반응",
        "코빼기도", "나몰라라",
        "직원 교육", "교육이 안", "교육을 안",

        # 컴플레인 예고
        "컴플레인", "항의", "민원", "본사에", "신고",
        "진상 부려", "한 소리",

        # 희소성/브랜드 가치 하락
        "흔하다", "흔해졌", "길거리", "아무나", "다 들고",
        "명품 아닌", "돈값",

        # 중고/리셀 가품
        "구구스", "중고", "리셀", "크림",
    ],

    "정책리스크": [
        # 가격 인상
        "가격인상", "가격 인상", "인상폭", "인상 폭",
        "또 올랐", "또 인상", "자꾸 올려", "계속 올려",

        # 환불/교환
        "환불", "교환거절", "환불불가",

        # 구매 제한
        "구매 제한", "1인1개",

        # 정책 일반
        "정책", "규정", "약관",

        # 고객 과실 전가
        "고객 책임", "고객책임", "과실", "소비자 과실",
        "책임 전가", "본인 과실",

        # 불량 판정 프로세스
        "본사", "감정", "불량 아니", "하자 아니",
        "프랑스", "판정", "인정 안", "책임 안",

        # 수리비
        "수리비", "무상수리", "유상",

        # 누락/미제공
        "보증서", "파우치", "정품 등록", "등록 놓쳐",
    ],
}

# 2. 분류 함수
# 우선순위 정의 (비즈니스 판단으로 조정 가능)
priority = ["품질리스크", "정책리스크", "운영리스크", "평판리스크"]

def classify_risk(text):
    scores = {category: 0 for category in risk_keywords}

    for category, keywords in risk_keywords.items():
        for keyword in keywords:
            if keyword in text:
                scores[category] += 1

    max_score = max(scores.values())

    if max_score == 0:
        return "미분류"

    # 동점이면 우선순위로 결정
    top_categories = [c for c, s in scores.items() if s == max_score]

    for p in priority:
        if p in top_categories:
            return p

df_risk["risk_category"] = df_risk["text"].apply(classify_risk)

In [9]:
# 1. 분류 결과 비율 확인
print(df_risk["risk_category"].value_counts())

# 2. 미분류 비율 확인
no_categorize = (df_risk["risk_category"] == "미분류").sum() / len(df_risk) * 100
print(f"미분류 비율: {no_categorize:.1f}%")

risk_category
미분류      955
운영리스크    491
품질리스크    175
평판리스크    158
정책리스크     81
Name: count, dtype: int64
미분류 비율: 51.3%


In [10]:
pd.set_option('display.max_colwidth', None)
# 3. 미분류 샘플 확인 (BERT fallback 전에 육안으로 먼저 봐)
df_risk[df_risk["risk_category"] == "미분류"]["text"].sample(50)

439                                                                                                                                                                                                                                                                                                                                                                       25년 된 베누아 어머니한테 물려받은 25년 넘은 베누아예요..! 요즘 베누이보다 더 둥근 모양이라고 하던데 맞나요??  저는 20대 중반인데 하고다님 조금 올드해보이려나요..?  밑에 베이지 스트랩으로 바꿔서 끼고 다니려고 하는데 혹시 은은한 느낌으로다가 스트랩 색 추천해주시면 감사하겠습니다!!
4387                                                                                                                                                                                                                  러브 힌지vs나사 다음 주 드디어 오리지널 러브 구매하러 가는데요.. 힌지가 스몰힌지랑 다르게 개선되었다고해서 당연히 힌지로 하려는데 최근에 러브 들인 분들 글보면 생각보다 많은 분들이 기존 버전을 고르신 것 같아요.  아무래도 문신템이라 그러신 것 같는데.. 저는 문신템으로 하려고 해도 힌지버전으로 고르려고 했거든요.  둘다 바로 구매할 수 있다는 전제하에 문신템이라는 전제하에 무엇을 추천하시는지 투표해주시면 구매할때 큰 도움이 될 것 같아요^^  +결론

In [11]:
df_risk[df_risk["risk_category"] == "미분류"]["text"].sample(50).to_excel("미분류50개.xlsx")

In [13]:
! pip install sentence_transformers
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. 한국어 사전학습 모델 로드
model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')

# 2. 카테고리 대표 문장 정의
category_descriptions = {
    "품질리스크": "도금이 벗겨지고 기스가 나고 변색되고 옐골화가 되고 광이 죽고 쇳덩이처럼 변하고 가품이 의심되고 시계가 느려지거나 멈추고 부품이 고장난다",
    "운영리스크": "웨이팅이 길고 까픈런이 힘들고 재고가 없고 배송이 지연되고 AS가 안 된다",
    "평판리스크": "직원이 불친절하고 싸가지 없고 응대가 최악이고 브랜드에 실망하고 불매한다",
    "정책리스크": "가격을 자꾸 인상하고 환불이 거절되고 수리비를 청구하고 구매를 제한한다"
}

# 3. 카테고리 임베딩 미리 계산
category_embeddings = {
    cat: model.encode(desc)
    for cat, desc in category_descriptions.items()
}

# 4. fallback 함수
def classify_unclassified(text):
    text_embedding = model.encode(text)

    similarities = {
        cat: cosine_similarity([text_embedding], [emb])[0][0]
        for cat, emb in category_embeddings.items()
    }

    return max(similarities, key=similarities.get)

# 5. 미분류만 fallback 적용
mask = df_risk["risk_category"] == "미분류"
df_risk.loc[mask, "risk_category"] = df_risk.loc[mask, "text"].apply(classify_unclassified)

# 6. 결과 확인
print(df_risk["risk_category"].value_counts())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/467M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

risk_category
운영리스크    1181
품질리스크     363
평판리스크     165
정책리스크     151
Name: count, dtype: int64


In [14]:
df_risk[df_risk["risk_category"] == "운영리스크"]["text"].sample(20)

959                                                                                                                                                                                       판교 현백 셀러 지방에 살고 애기들 키우느라 평일밖에 시간이 안되는데 등원시키고 출발해도 오픈런은 못할것같아서요 ㅠㅠ 미리 예약하고 당일 러브팔찌 구매예정인데  셀러분 연락처 공유 가능할까요 🥹
5010             까르띠에 앵끌루팔찌 엄마가 엄마 자신에게 주는 선물로 구매하셨고 ^^   세르펜티가 딱 어울리셨는데 뱀이 본인 띠랑(돼지띠 뱀은 상극이라며 ㅋㅋ)은 안 맞으신다고 두개 레이어드 착용템으로 앵끌루 오리지널을 며칠 전에 구매하셨는데  집에 와서 보니 아무래도 너무 영해 보인다고 앵끌루 다이아로 교환해야겠다고 하시네요 ㅋ 근데 가격차이가 ㅠ 그래도 다이아로 구매해야 후회를 안하실까요 너무 영해 보이지 않으실지? 아니면 나이가 있으시니  세르펜티로 가는걸 강력 추천해야 할까요?조언 부탁드려요~ ^^
3320                                                                                                                                                                                                                                                             러브 사이즈 도와주세요 손목 둘레 14, sm 16호 사이즈입니다.
2034                                                                                                               

In [15]:
df_quality = df_risk[df_risk["risk_category"] == "품질리스크"]
df_operation = df_risk[df_risk["risk_category"] == "운영리스크"]
df_reputation = df_risk[df_risk["risk_category"] == "평판리스크"]
df_policy = df_risk[df_risk["risk_category"] == "정책리스크"]

print(len(df_quality), len(df_operation), len(df_reputation), len(df_policy))

363 1181 165 151


In [16]:
# 각 버킷에서 키워드가 하나도 없는 글 제거
# 즉 rule-based에서 키워드로 잡힌 글만 남기기

def has_keyword(text, category):
    keywords = risk_keywords[category]
    return any(kw in text for kw in keywords)

df_quality_clean = df_quality[df_quality["text"].apply(
    lambda x: has_keyword(x, "품질리스크"))]

df_operation_clean = df_operation[df_operation["text"].apply(
    lambda x: has_keyword(x, "운영리스크"))]

df_reputation_clean = df_reputation[df_reputation["text"].apply(
    lambda x: has_keyword(x, "평판리스크"))]

df_policy_clean = df_policy[df_policy["text"].apply(
    lambda x: has_keyword(x, "정책리스크"))]

print(len(df_quality_clean), len(df_operation_clean),
      len(df_reputation_clean), len(df_policy_clean))

175 491 158 81


In [63]:
!pip install bertopic -q
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer
import random
import numpy as np

embedding_model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')

stopwords = [
    # 자음/모음 단독 (ㄱㄱ, ㅠㅠ 등)
    'ㄱ', 'ㄴ', 'ㄷ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅅ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ',
    'ㄱㄱ', 'ㄱㄱㄱ', 'ㄱㄱㄱㄱ', 'ㅈㄴ', 'ㅈㄴㄱ',
    'ㅋㅋ', 'ㅋㅋㅋ', 'ㅎㅎ', 'ㅎㅎㅎ', 'ㅠㅠ', 'ㅠㅠㅠ', 'ㅜㅜ',

    # 까르띠에 브랜드명 (토픽 구분에 도움 안 됨)
    '까르띠에', '깔띠', '까르', '깔띠에',

    # 의미없는 부사/형용사
    '너무', '혹시', '하고', '제가', '저는', '같아요',
    '있어요', '싶어요', '했는데', '그냥', '많이',

    # 추가
    '정말', '진짜', '완전', '약간', '조금', '좀',
    '이제', '아직', '벌써', '드디어', '일단',
    '안녕하세요', '감사합니다', '감사해요', '부탁드려요',
    '궁금해요', '궁금합니다', '여쭤봐요', '올려요',
    '스몰', '미디움', '오리지널', '라지',  # 사이즈 표현

    # 추가2
    '더', '잘', '넘', '안', '것', '또', '글',
    '이', '가', '은', '는', '을', '를', '에', '의', '로', '으로',
    '하는', '되는', '있는', '같은', '이런', '그런',
    '정도', '경우', '때문', '부분',

    #제품명   
    '러브', '러브팔찌', '러브링', '러브브레이슬릿',
    '세르펜티', '클래쉬', '다무르', '앵끌루', '저스트앵끌루',
    '트리니티', '탱크', '탱머', '탱머스트', '산토스',
    '발롱블루', '발렉스', '팬더', '베누아',
    '팔찌', '반지', '목걸이', '시계', '귀걸이',  # 제품 카테고리
    '골드', '화골', '로골', '스틸', '가죽',  # 소재
    '다이아', '다이아몬드',

    # 기존 불용어에 추가
    '사이즈', '매장', '매장에서', '구매', '구입', '오늘',
    '분', '다들', '같이', '저도', '제',
    '1', '2', '3', '4', '5', '6', '7', '8', '15', '16', '17', '18' # 숫자/사이즈
    'ec',  # 오타/노이즈
    '떴어요', '근데', '다',

    # 불용어 추가
    '반클', '오닉스', '화이트골드', '프랑세즈', '풀파베',  # 제품명
    'ec', 'ㅠ', '있을까요', '했어요', '하나', '나을까요',  # 노이즈
]

vectorizer = CountVectorizer(
    stop_words=stopwords,
    token_pattern=r"(?u)\b\w+\b"  # 한글 토큰 인식
)

def run_bertopic(df_subset, category_name, nr_topics):
    docs = df_subset["text"].tolist()

    topic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer,
        nr_topics=nr_topics,
        min_topic_size=10,
        language="multilingual"
    )

    topics, probs = topic_model.fit_transform(docs)

    print(f"\n=== {category_name} ===")
    print(topic_model.get_topic_info())

    return topic_model, topics, probs

# 각 카테고리별 실행
model_quality, topics_q, probs_q = run_bertopic(df_quality_clean, "품질리스크", nr_topics=4)
model_operation, topics_o, probs_o = run_bertopic(df_operation_clean, "운영리스크", nr_topics=6)
model_reputation, topics_r, probs_r = run_bertopic(df_reputation_clean, "평판리스크", nr_topics=3)
model_policy, topics_p, probs_p = run_bertopic(df_policy_clean, "정책리스크", nr_topics=3)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: snunlp/KR-SBERT-V40K-klueNLI-augSTS
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== 품질리스크 ===
   Topic  Count             Name  \
0     -1     52    -1_옐골_여유_딱_vs   
1      0     78  0_기스_레이어드_계속_미니   
2      1     28  1_교환_다시_폴리싱_시계가   
3      2     17  2_화골이_변색_체인이_거의   

                                    Representation  \
0      [옐골, 여유, 딱, vs, 시크님들, 골라주세요, 지금, 뱅글, 미니, 링]   
1   [기스, 레이어드, 계속, 미니, 분들, 뱅글, 인상전, 이번에, 사진펑, 같아서]   
2     [교환, 다시, 폴리싱, 시계가, 바로, 제품, 보니, 1회, 와서, 네크리스]   
3  [화골이, 변색, 체인이, 거의, 5모티브, 팔찌는, 화이트, 됐는데, 남자, 스윗]   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [64]:
model_quality.get_representative_docs(2)  

['5모티브 팔찌 추천 해주세요💛🖤🤍 악세사리는 목걸이 광(?)이라 목걸이만 주구장창 사고 ㅠㅠ 팔찌는 하나도 없는데요,, 반클리프 제품으로는 -빈티지 타이거즈아이 알함브라 목걸이 -빈다 옐골 목걸이 -마오펄 스윗 목걸이 -오닉스 스윗 이어링  이렇게 있는데, 5모티브 팔찌를 산다면 어떤걸 추천하시나요?  셀러분은 빈다옐골이랑 오닉스 스윗 이어링이 있으니 오닉스 5모팁 추천 해주시긴 했는데… 마오펄로 하면 너무 제각각일까요?  기요셰 5모팁은 기스 이슈로 제외했고, 기요셰+빈다 조합은 가격 이슈로 제외했어요ㅠㅠ  아니면…. 5모티브 말고 까르띠에 앵끌루sm이나 러브 미듐 팔찌는 어떨까요?🤔',
 '금색 줄 변색되는거요 반클 목걸이 한5년 됐는데 많이 끼지는 않았어요. 매장서세척도 거의 안한거같구요(말카) 그냥 보통의 금목걸이처럼 보관은 잘하지만 그다지 유지관리는 신경 안썼어요ㅠ그래도 되는게 금아닌가요???  그러다 어느날 매장가서보니 제것이 정말 노~~랗더라구요 그럼에도 또 목걸이를 샀어요 팔찌도 사고.(오닉스,자개) 새거는 이렇게 레몬색이구나~~하면서요;;  그러다가 지금은 자개 귀걸이도 사고싶은데… 노래지는거 생각하면 안사는게 나을까요? 귀걸이는 원석이랑 붙어서 세척도 제대로 안될거같은데.. 그것때문에 사기가 망설여져요 금액도 올라서 700…  까르띠에도 핑크골드로만 있는데 이것도ㅠ 색빠지고 노란금 된다는데 매장가서 비교안해봐서 모르겠어요. 이또한 잘안끼지만 당연히 관리는 안했어요. 금이니까…?!  차라리 까르띠에 시계에 금도 색이 변하나여?? 옐로>누런금색으로, 핑골>노란금으로??  아 그럼 반클도 변하고, 까르띠에도 변한다면 사기가 싫으네요ㅠㅠ 지금 고려중인건, 반클 자개 빈티지 귀걸이냐, 탱크루이냐ㅜ고민중이거든요. 변하는거 싫은데 둘다 색변하나요?? 백화점자주안가니까 매달 세척못햐요;',
 '까르띠에 고민있어요ㅠ 어제 까르띠에 클래쉬 목걸이와 반지를 사러 매장에 갔어요.. 목걸이는 괜찮았는데 반지가 제 손에 아무리 봐도 예쁜 느낌이 없고 셀러분께서 검

In [65]:
model_operation.get_representative_docs(2)

['탱머 가죽 라지 드디어 겟했는데요.  까르띠에에 물어보니 스틸 스트랩 가격이 150만원 선이라는데 실화인가요? 6~70으로 알고 있었는데 가죽스트랩과 스틸스트랩이 차이가 이렇게 컸나요?  탱머 가죽 라지 540 탱머 스틸 라지 600  가죽 스트랩 모름 스틸 스트랩 150(까르띠에 카톡상담으로 확인)  당연히 스틸 라지 사고 가죽 스트랩 별도 구매가 정석이라 그렇게 구매하려고 했지만 언제 또 입고될지 몰라 가죽 라지라도 샀습니다만 그냥 보내줘야할지 고민이네요..',
 '탱크머스트 가죽 구매 또는 가죽줄 교체하신 분 중에 길이 연장? 추가 하신분 계신가요? 운좋게 탱머 스몰 스틸로 공홈에서 구입했눈데요~ 가죽줄 구매하려 매장에 갔눈데 송아지 가죽줄 1cm 추가 연장하면 좋겠다 해서  별생각 없이 그러고 왔는데 이게 맞나 싶어요  참고로 제 팔목  15cm 이고 탱머 스몰 스틸로 구입했어요  핀버클포함 해서 49마넌 정도 비용 나왔는데 다른분들은 40초반대라 하시고 ..  연장? 1cm 추가 하신분 계신가요?  손목 15이신 시크님들중에 가죽 버클부분 많이 짧둥한지  의견 묻고 싶습니다',
 '워치 스틸스트랩 주문도 원래 일반 제품 상담이랑 똑같이 기다리나요?ㅠ  메종은 평소 동선이 아닌지라 스트랩 찾으러 메종가기 귀찮아서 백화점에서 하려고 시계살 때 안했더니 스트랩 주문하기도 만만치않네요. 예전에 보기론 스트랩 주문은 AS로 쳐서 금방 된다고 본 것 같은데 잘못알았나봐여😭😭😭😭  이것도 웨이팅이 하염없네요.... 가죽이면 모르겠는데 스틸은 걍 주문만 하면 되는데도 기다려야하다니ㅠ 이럴 줄 알았으면 걍 시계 살때 스트랩 주문까지 할 걸 그랬어요.  스트랩 하나 사려고 기다리기 너무나 시간 아까운 것... 메종에서 탱머 사시는 분들 계시다면 꼭 시계사면서 스트랩 같이 주문해두세요....']

In [66]:
model_operation.get_representative_docs(0)

['클래쉬 팔찌 뱅글 구매했는데 사이즈 고민요.. 팔찌 사기전까지 구매후기 거의 다읽어본거같아요.. 온라인 구매하고 매장픽업 방금하고왔는데요!  우선 제 손목사이즈는 13.9-14cm정도여서 클래쉬 스몰 팔찌 14호로 구매했어요. 몇년전 다른 도시 매장에서나 이번 매장에서나 모두, 미국 직원 언니들은 14호가 맞는거 같다고 했어서 온라인으로 바로 구매했는데..  막상 차보니 좀 여유가 너무 없나? 싶어서 고민되요. 일단 손목에 끼지는 않고, 마지막 사진에 팔 위치까지 쭉 내려오는데, 하나 더 큰 15호로 해서 여유있게 차야지 레이어드할때 이쁜지 고민되네요..   지금 구매한 14호가 네번째 사진처럼 여유가 있긴 하고요. 15호 찼을때는 손가락 세개정도 들어갈 정도로 남고 팔 중간까지 훌렁 내려와서, 미국 매장언니가 15는 크다고 했었어요. 사실 이언니들 워낙 교환 귀찮아서 그냥 말하는거일수도 있고요 ㅠㅠ 시크님들 후기보니 손목 제 사이즈인데 14호 사신분들도 아무도 안보이네요..! 최소 15는 사신거같아요 다들  시크님들 눈엔 사이즈 어때보이나요? 레이어드 여러개 할때는 좀 여유로운 뱅글 사이즈가 멋스러운가요? 아니면 지금처럼 맨 손목에 잘 맞는게 이쁠까요. 종일 고민되서 의견 구합니다아 ㅠㅠ!!! 마지막에 15호 차본 사진도 있으니 둘중에 어느사이즈가 나을지 의견 부탁드려요!!  아래사진은 14호',
 '시계 추천 시계 잘 안하는 편이긴 해요.  출근할때는 애플워치를 주로 하고, 그 외 모임이나 여행때는 팔찌를 주로 껴요.  그래서 결혼때 고민하다가 괜히 가방을 2개 했는데 좀 후회되네요.  쥬얼리를 더 하거나 시계를 할걸....  가방은 정말 가격대비 사용 빈도가 현저히 낮네요.  악세사리는 출근때도 잘 하게 되는데  가방은 너무 티가 나기도 하고 조심스러워서 휘뚤 마뚤 쓰는 가방만 주로 들게 되는 것 같아요.   결혼하고 물욕이 좀 사라져 명품 잊고 있었는데,  최근 까르띠에 인상 대란 보고 더 늦기 전에 시계 하나는 사둬야 겠다 싶어서 고민하기 시작했어

In [67]:
model_operation.get_representative_docs(3)

['러브팔찌 2월 디파짓 한거 받아왔어요 (사진펑) 신세계 영등포에서 구매했어요 !  인상 직전날에 오픈런해서 9시간인가 기다리고 주문했는데 막상 받아보니 너무너무 예뻐요  인상전에 막차타고 구매하길 잘했다 싶습니다 .. 🥺  까르띠에는 항상 웨이팅 제일 없는 압갤에서만  구매하고 영등포는 처음인데  정말 세상에서 제일 친절한  센스있는 셀러님한테 구매해서  너무너무 행복한 마음으로 사고 돌아갑니다  올해 팔찌를 두개나 사서 당분간 쇼금이지만  너무 기분 좋아요 팔찌 볼때마다 행복할것같아요 !!',
 '앵끌루링 다이아! sm 오늘 구입한 따끈 후기 (웨이팅 정보) 안녕하세요 도움 많이 받아서 오늘 구매후기 남겨요 ! 영등포 신세계 탐스큐ㅔ어로 오후 3시 정각에 웨이팅 걸어서 오후 6:30 에 들어갔어요 ㅜㅜ 구매하실 분들은 주말 보단 평일 노리셔야 할듯해요 2월 인상 코앞 때는 오픈런 아님 못들어가실 정도의 화력 같아요! 본점이나 강남권 아닌 영등포 평일인데 이정도 였우니까요 ㅜㅜㅜㅜ  진짜 인상 코앞이라고 해주셨고 사이즈별로 컬러별로 디자인별로 다 착용해보고 가장 어울리는 앵끌류 sm 다이아로 가져왔어요!  사실 노다이아는 189고 다이아는 398 이라서 ㅜ 두배나 더 태우는 게 맞나.: 했는데 앵끌루는 확실히 설탕ㅇ ㅣ 뿌려져야 예쁜듯해요',
 '인상전 공홈 구매건 사이즈교환신청 했어요~  1/24일에 공홈에서 [러브미듐 로골 17사이즈] 구매하였을 당시는 2/4일 배송예정이였습니다~  그러나 다른 분들처럼 출고지연 문자를 받고 (따로 3~4개월 걸린다는 문자는 없었어요 ~), 하염없이 기다리던 중,  2/11일에 배송시작한다고 까르띠에 컨택센터에서 연락이 오더니 카톡으로 발렉스 배송 문자가 오더라구요~  설레는 마음으로 2/12일에 수령하였으나..!!  제 손목에 17은 너무 정직하게 맞아보이더라구요 .. 🤧  그래서 사이즈 교환이 되는지 2/13일 오늘 바로 전화상담을 했습니다~!  (된다고 상담은 받았으나, 카페내 글에서 보면 상담하셨던 부분이 다들 

In [71]:
model_reputation.get_representative_docs(0)

['러브 미듐 사이즈 조언 부탁드려요..!!교환하러 가야할까요ㅠㅠ   손목 뼈 기준 13.5이고 손목뼈 위 가장 얇은부위가 12.7정도 나와요..!!   매장에서 셀러님이 16호는 돌아가서 불편할거라고 하셔서 15호로 가져왔는데 너무 알맞게(?) 맞는거 같아요 ! 교환하러 가야할지 고민입니다   활동하기에는 15호가 편한거 같은데… 16호는 세로로 끼기도 해서 정리해줘야 하고 좀 불편할거 같더라두요..!!러브 크게 사신분들 안불편하시나요?  사진펑   결국 16호 교환했어요^^모두들 의견주셔서 감사드려요~',
 '까르띠에 러브팔찌 디파짓 넣고왔는데 18사이즈가 큰거같아 봐주세요ㅠ  러브 미듐 사이즈로 정하고 미듐은 사이즈가 없어 오리지널로 17끼고 18도 괜찮길래 18로 정했어요  지금 집와서 재보니 17이 나은거같아 사이즈를 잘못정했나 후회돼요ㅠㅠ  디파짓하면 사이즈 변경도 안된다한거같은데 이것도 검색하면 셀러님 재량인지 말들이 다르네요🥲  매장에서 미듐17 사이즈가 없어 오리지널 17로 착용했어요  처음에 옷을 제대로 안걷고 봐서 그런지 팔꿈치쪽으로 이쁘게 내려오려면 18이 맞겠다 생각했어요 문제는 손목 손등 앞으로 쏠리는거 계산하면 18사이즈는 손바닥까지와서 착용감이 안좋을거 같은거에요ㅠㅠ     오리지널17, 미듐19인데 19는 훌러덩 너무커서 느낌만 봤어요   막상 집에서 제가 17이랑 18 만들어 재보니 오른손목이 더 얅더라고요ㅠㅠ  손목이 14.9-15cm 정도고 현재 살쪄서 나중까지 감안해도 지금 사이즈에 맞추면돼요  팔뚝쪽으로 내려오는건 괜찮은데  제가 팔목뼈도 거의 안튀어나와서 손등 앞으로 너무 떨어질거같아요  인상되고 미듐사이즈 여유있을때 사이즈 잘 체크하고 살걸싶고ㅠㅠ 18사이즈해놓고 너무 걱정돼요🥲',
 '러브 팔찌 사이즈 교환?? ㅠ 어제 러브팔찌 오리지널로 15호 구입했어요 . 아직 포장은 뜯지 않은 상태로 사이즈가 애매해서 여쭤봐요 . 15호는 살짝 여유 있는 느낌이었고 16호는 팔을 내릴때 손목 아래로 내려와서 고민 끝에 15 선

In [69]:
model_policy.get_representative_docs(1)

['인생 처음이자 마지막일 깔띠에~ 안녕하세요~~  백화점을 아주 맨날맨날 가는데  깔띠에는 오늘이 첫방문이었네요 ㅎㅎ  애둘맘이자 워킹맘인 전 인상 전에 오픈런 성공할 자신이 없어서 백화점 vip혜택인 매장 예약을 처음으로 해봤습니다 ㅎㅎㅎ   원래는 스몰다이아 사이즈의 앵끌루 팔찌를 생각해보고 착용했는데 남편이 두꺼운것도 차보라고 하더라구요 ㅋㅋㅋ   가격이고 뭐고 아무것도 몰랐던거죠 ㅋㅋㅋ  그렇게 앵끌루 오리지널 하프다이아를 차게 되었고 영영 제꺼가 되었답니다 :)  쫌 너무 과하지 않나 싶었는데 전신거울엔 딱 이쁘더라구요~~  오늘이 생애 첫 까르띠에 방문이었는데  생애 마지막 방문으로 할라구요 ㅠㅡㅠ   (근데 클래쉬드 귀걸이도 차봤는데 남편이 넘 이쁘다구 ㅠㅠ 하..)  인상 소식 알려주신 시크님들 감사합니다~! 전체적인 인상률은 5%정도인데 시계인상폭이 크다고 하더라구요! 다들 이쁜거 겟하시길~  근제 사진이쁘게 찍기 어렵네요 ㅜ',
 '반지 사라마라 해주세요 ㅠ  최근에 까르띠에 클래쉬 반지를 구입했어요 반지 사이즈교환으로 트리니티를 껴봤더니 다들 스몰보다는 클래식 추천해주셨어요 ㅠㅠ 근데 최근에 목걸이 반지까지 샀더니.. 200선이 적당한것같은데 이번에 또 인상한다니... 문의드립니다.   회사를 다니는데 여자사장이랑 둘만 있어서.. 평소에 악세서리나 이런걸 차고오면 자연스럽게 눈길이 가요.. 근데 제가 진짜 안하다가 최근에 눈을 떠서.. 한번도 악세서리를 안하다가 클래쉬반지를 차려니.. ㅋㅋㅋㅋㅋ 괜히 제가 눈치가 보이고... 사실 사장님은... 금은방에서 구입하셔서 ㅋㅋ 클래쉬는 주말에는 데일리로 끼는데 회사에선 도저히 안되겠어요!!! 이건 진짜 불가.. 그래서 트리니티 스몰은 눈에 띄지도않지만 이쁘기도하고.. 제만족일것같은데 다들 사이즈 클래싱아님 후회한다고해서 고민입니다 금액도 금액이구요 ㅠㅠ 근데 까르띠에 인상은 한다고 하지.. 마음이 갈팡지팡이예요 ㅋㅋ  그래서 그냥 가드링 같은 반클리프 비즈링을 구매 한다.. 하나만차도 심플하고 

품질: 기스우려_내구성불안 / 외관불량_하자인정거부
운영: 웨이팅_오픈런 / 공홈재고부족 / 재고부족_스트랩입고대기
평판: 셀러불친절_응대불량
정책: 교환환불정책불만 / 가격인상불만

In [72]:
print("품질리스크 전체:", len(df_quality))
print("품질리스크 clean:", len(df_quality_clean))
print("품질리스크 -1:", sum(1 for t in topics_q if t == -1))

print("운영리스크 전체:", len(df_operation))
print("운영리스크 clean:", len(df_operation_clean))
print("운영리스크 -1:", sum(1 for t in topics_o if t == -1))

print("평판리스크 전체:", len(df_reputation))
print("평판리스크 clean:", len(df_reputation_clean))
print("평판리스크 -1:", sum(1 for t in topics_r if t == -1))

print("정책리스크 전체:", len(df_policy))
print("정책리스크 clean:", len(df_policy_clean))
print("정책리스크 -1:", sum(1 for t in topics_p if t == -1))

품질리스크 전체: 363
품질리스크 clean: 175
품질리스크 -1: 52
운영리스크 전체: 1181
운영리스크 clean: 491
운영리스크 -1: 190
평판리스크 전체: 165
평판리스크 clean: 158
평판리스크 -1: 15
정책리스크 전체: 151
정책리스크 clean: 81
정책리스크 -1: 25


In [73]:
# 각 카테고리별 토픽 컬럼 추가
df_quality_clean["topic"] = topics_q
df_operation_clean["topic"] = topics_o
df_reputation_clean["topic"] = topics_r
df_policy_clean["topic"] = topics_p

# 토픽 이름 매핑
quality_topic_map = {
    -1: "노이즈",
    0: "기스우려_내구성불안",
    1: "외관불량_하자인정거부",
    2: "노이즈",
}

operation_topic_map = {
    -1: "노이즈",
    0: "노이즈",
    1: "웨이팅_오픈런",
    2: "재고부족_스트랩입고대기",
    3: "노이즈",
    4: "공홈재고부족",
}

reputation_topic_map = {
    -1: "노이즈",
    0: "노이즈",
    1: "셀러불친절_응대불량",
}

policy_topic_map = {
    -1: "노이즈",
    0: "교환환불정책불만",
    1: "가격인상불만",
}

# 매핑 적용
df_quality_clean["topic_name"] = df_quality_clean["topic"].map(quality_topic_map)
df_operation_clean["topic_name"] = df_operation_clean["topic"].map(operation_topic_map)
df_reputation_clean["topic_name"] = df_reputation_clean["topic"].map(reputation_topic_map)
df_policy_clean["topic_name"] = df_policy_clean["topic"].map(policy_topic_map)

# 전체 합치기
df_bertopic = pd.concat([
    df_quality_clean,
    df_operation_clean,
    df_reputation_clean,
    df_policy_clean
])

# 노이즈 제외
df_bertopic = df_bertopic[df_bertopic["topic_name"] != "노이즈"]

print(df_bertopic["topic_name"].value_counts())
print(f"\n최종 분석 데이터: {len(df_bertopic)}개")

topic_name
셀러불친절_응대불량      117
웨이팅_오픈런          85
기스우려_내구성불안       78
재고부족_스트랩입고대기     50
가격인상불만           32
공홈재고부족           29
외관불량_하자인정거부      28
교환환불정책불만         24
Name: count, dtype: int64

최종 분석 데이터: 443개


In [74]:
# 중복 텍스트 있는지 확인
print(df_risk["naver_article_id"].duplicated().sum())

0


In [75]:
# 1. rule-based + fallback 결과 (1860개, risk_category 컬럼 있음)
# 2. bertopic 결과 (443개, topic_name 컬럼 있음)

# bertopic 결과에서 필요한 컬럼만 추출
df_topic_result = df_bertopic[["naver_article_id", "topic_name"]]

# 원본 df에 topic_name 합치기
df_final = df_risk.merge(df_topic_result, on="naver_article_id", how="left")

# 확인
print(df_final.shape)
print(df_final["risk_category"].value_counts())
print(df_final["topic_name"].value_counts(dropna=True))

# 저장
df_final.to_excel("bertopic3.xlsx", index=False)

(1860, 25)
risk_category
운영리스크    1181
품질리스크     363
평판리스크     165
정책리스크     151
Name: count, dtype: int64
topic_name
셀러불친절_응대불량      117
웨이팅_오픈런          85
기스우려_내구성불안       78
재고부족_스트랩입고대기     50
가격인상불만           32
공홈재고부족           29
외관불량_하자인정거부      28
교환환불정책불만         24
Name: count, dtype: int64


In [76]:
# text가 결측인 게시글 확인
print(df_final[df_final['text'].isna()]['content'].head(3))

# 빈 문자열도 같이 확인
print((df_final['text'] == '').sum())  # 빈 문자열 개수
print(df_final['text'].isna().sum())   # 진짜 NaN 개수

Series([], Name: content, dtype: str)
0
0


In [77]:
df_final.isnull().sum()

Unnamed: 0                  0
naver_article_id            0
writer_nickname             0
released_at                 0
view_count                  0
title                       0
like_count                  0
comment_count               0
comments                   81
content                     0
clean_content               2
log_view_count              0
log_like_count              0
log_comment_count           0
text                        0
is_cartier                  0
is_product                  0
deleted_comment_count       0
deleted_comment_ratio       0
label                       0
pred_label                  0
pred_prob                   0
se_text                     0
risk_category               0
topic_name               1417
dtype: int64